# Analysis: parameter identifiability and fit quality

This notebook loads all artifacts from `results/first_estimation/` (produced by
`first_estimation.ipynb`) and generates publication-quality figures.  It does NOT
import the estimator, re-run any steady-state solve, or require CasADi / Pyomo /
ipopt -- only numpy, pandas, and matplotlib are needed.

All figures are saved to `results/first_estimation/figures/`.


In [1]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_REPO_ROOT = os.path.abspath(".")
sys.path.insert(0, os.path.join(_REPO_ROOT, "src"))

import glyco_plots as gp

RESULTS_DIR = os.path.join(_REPO_ROOT, "results", "first_estimation")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

res = gp.load_results(RESULTS_DIR)
print("Loaded artifacts from:", RESULTS_DIR)
for k, v in res.items():
    if v is None:
        print("  %-30s None" % k)
    elif isinstance(v, pd.DataFrame):
        print("  %-30s DataFrame %s" % (k, v.shape))
    elif isinstance(v, pd.Series):
        print("  %-30s Series (len=%d)" % (k, len(v)))
    elif isinstance(v, dict):
        print("  %-30s dict (keys=%s)" % (k, list(v.keys())))
    else:
        print("  %-30s %s" % (k, type(v).__name__))

Loaded artifacts from: /Users/gabbi/Desktop/Repos/biosistemas/HybridKinetics-IIQ3733/results/first_estimation
  theta_init                     Series (len=37)
  theta_fitted                   None
  theta_init_sources             DataFrame (37, 2)
  covariance                     None
  correlation                    None
  confidence_intervals           None
  fim                            DataFrame (37, 37)
  predictions_init               DataFrame (22, 18)
  predictions_fitted             None
  real                           DataFrame (22, 18)
  rmse_init                      dict (keys=['met', 'flux', 'data_norm', 'weighted'])
  rmse_fitted                    None
  sensitivity                    dict (keys=['KO02', 'KO03', 'KO04', 'KO05', 'KO07', 'KO08', 'KO10', 'KO11', 'KO12', 'KO13', 'KO14', 'KO15', 'KO16', 'KO17', 'KO18', 'KO19', 'KO20', 'KO21', 'KO22', 'KO23', 'KO24', 'RF03'])
  manifest                       dict (keys=['seed', 'sigma_log10', 'conditions', 'n_conditions', 

## Run manifest


In [2]:
manifest = res.get("manifest")
if manifest:
    print("Run manifest:")
    for k, v in manifest.items():
        print("  %-20s %s" % (k, v))
else:
    print("No manifest found.")

Run manifest:
  seed                 0
  sigma_log10          0.5
  conditions           ['KO02', 'KO03', 'KO04', 'KO05', 'KO07', 'KO08', 'KO10', 'KO11', 'KO12', 'KO13', 'KO14', 'KO15', 'KO16', 'KO17', 'KO18', 'KO19', 'KO20', 'KO21', 'KO22', 'KO23', 'KO24', 'RF03']
  n_conditions         22
  free_params          ['v_max_1', 'Ka1_1', 'Ka2_1', 'Ka3_1', 'K_g6p_1', 'Ks_g6p_pgi', 'Kp_f6p_pgi', 'kcat_f_2', 'Ks_f6p_3', 'Ks_atp_3', 'Kp_fbp_3', 'Kp_adp_3', 'kcat_f_3', 'Ks_fbp_4', 'Kp_g3p_4', 'Kp_dhap_4', 'kcat_f_4', 'kcat_f_5', 'Ks_dhap_5', 'Kp_g3p_5', 'kcat_f_6', 'Ks_g3p_6', 'Ks_pi_6', 'Ks_nad_6', 'Kp_pgp_6', 'Kp_nadh_6', 'kcat_f_7', 'Ks_pgp_7', 'Ks_adp_7', 'Ks_3pg_7', 'Ks_atp_7', 'kcat_f_8', 'Ks_3pg_8', 'Ks_2pg_8', 'kcat_f_9', 'Ks_2pg_9', 'Ks_pep_9']
  n_free_params        37
  obj_value            None
  ipopt_available      False
  timestamp            2026-06-25T05:31:01.246404Z


## RMSE: initial vs. fitted

The `data_norm` RMSE normalizes each output by its mean measured value across
conditions and averages, giving a dimensionless aggregate that is comparable
across outputs of very different magnitudes.  Values >> 1 indicate the model
misses by more than the mean signal.

The `weighted` RMSE uses the measurement sigma (same as the objective function)
and is in sigma units: ~1 means the fit sits at the measurement noise floor.


In [3]:
rmse_init = res.get("rmse_init")
rmse_fitted = res.get("rmse_fitted")

if rmse_init:
    tbl = {"init": rmse_init}
    if rmse_fitted:
        tbl["fitted"] = rmse_fitted
    rmse_df = pd.DataFrame(tbl)
    print("RMSE comparison:")
    print(rmse_df.round(4).to_string())

    fig, ax = gp.plot_rmse_summary(
        rmse_init, rmse_fitted,
        savepath=os.path.join(FIGURES_DIR, "rmse_summary.png")
    )
    print("Figure saved: rmse_summary.png")
else:
    print("No RMSE data found.")

RMSE comparison:
               init
met          1.7926
flux         6.7565
data_norm   18.2298
weighted   139.8396


Figure saved: rmse_summary.png


## Parameter values: initial vs. fitted

Log-scale comparison.  Large discrepancies between init and fitted values point
to parameters where the literature / Bar-Even prior is far from the optimal.
Parameters that remain near the initial value are either insensitive or
already at their optimum.


In [4]:
theta_init = res.get("theta_init")
theta_fitted = res.get("theta_fitted")

if theta_init is not None:
    fig, ax = gp.plot_theta_init_vs_fitted(
        theta_init, theta_fitted,
        savepath=os.path.join(FIGURES_DIR, "theta_comparison.png")
    )
    print("Figure saved: theta_comparison.png")
else:
    print("theta_init not found.")

Figure saved: theta_comparison.png


## Parameter correlation matrix

High |r_ij| (> 0.9) identifies pairs of parameters that cannot be independently
estimated from the available data -- one of each pair is a candidate to fix in a
reduced model.  The structure of correlated clusters also reveals which enzymatic
subsystems share informational content in the Ishii dataset.

**Interpretation guide:**
- Block-diagonal structure -> parameter groups are independently identifiable.
- Off-diagonal blocks -> cross-reaction parameter coupling.
- If the correlation matrix is not available (no ipopt), the FIM spectrum below
  still provides an a-priori view of identifiability.


In [5]:
corr_df = res.get("correlation")

if corr_df is not None:
    fig, ax = gp.plot_correlation_heatmap(
        corr_df,
        title="Parameter correlation (post-fit)",
        savepath=os.path.join(FIGURES_DIR, "correlation_heatmap.png")
    )
    print("Figure saved: correlation_heatmap.png")

    # Identify highly correlated pairs (|r| > 0.9)
    high_corr = []
    params = list(corr_df.index)
    arr = corr_df.to_numpy(dtype=float)
    for i in range(len(params)):
        for j in range(i + 1, len(params)):
            if abs(arr[i, j]) > 0.9:
                high_corr.append({
                    "param_i": params[i],
                    "param_j": params[j],
                    "r_ij": round(arr[i, j], 4),
                })
    if high_corr:
        hc_df = pd.DataFrame(high_corr).sort_values("r_ij", key=abs, ascending=False)
        print("Strongly correlated pairs (|r| > 0.9):")
        print(hc_df.to_string(index=False))
        print("\nThese pairs are candidates to fix in the reduced model.")
    else:
        print("No strongly correlated pairs (|r| > 0.9) found.")
else:
    print("Correlation matrix not available (estimation not run or ipopt absent).")
    print("See FIM spectrum below for a-priori identifiability.")

Correlation matrix not available (estimation not run or ipopt absent).
See FIM spectrum below for a-priori identifiability.


## Confidence intervals


In [6]:
ci_df = res.get("confidence_intervals")

if ci_df is not None:
    fig, ax = gp.plot_ci_errorbars(
        ci_df,
        savepath=os.path.join(FIGURES_DIR, "confidence_intervals.png")
    )
    print("Figure saved: confidence_intervals.png")

    # Parameters with CV% > 100% are poorly identifiable.
    if "cv_percent" in ci_df.columns:
        poor = ci_df[ci_df["cv_percent"] > 100].sort_values("cv_percent", ascending=False)
        print("\nParameters with CV% > 100%% (poorly identified):")
        print(poor[["theta", "std_err", "cv_percent"]].round(2).to_string())
else:
    print("Confidence intervals not available.")

Confidence intervals not available.


## Fisher Information Matrix: eigenvalue spectrum

The FIM eigenvalue spectrum reveals the intrinsic identifiability of the parameter
space from the data.  A steep drop (many near-zero eigenvalues) means the data do
not constrain all parameter combinations: only the directions corresponding to the
dominant eigenvalues can be estimated reliably.

Concretely: FIM rank < 37 -> at least (37 - rank) parameter combinations are
non-estimable and should be fixed in the reduced model.


In [7]:
fim_df = res.get("fim")

if fim_df is not None:
    fig, ax = gp.plot_fim_spectrum(
        fim_df,
        savepath=os.path.join(FIGURES_DIR, "fim_spectrum.png")
    )
    print("Figure saved: fim_spectrum.png")

    eigvals = np.sort(np.linalg.eigvalsh(fim_df.to_numpy(dtype=float)))[::-1]
    rank_est = int(np.sum(eigvals > 1e-6 * eigvals[0]))
    print("FIM shape:", fim_df.shape)
    print("FIM effective rank (eigval > 1e-6 * max):", rank_est)
    print("Non-estimable combinations:", fim_df.shape[0] - rank_est)
else:
    print("FIM not available.")

Figure saved: fim_spectrum.png
FIM shape: (37, 37)
FIM effective rank (eigval > 1e-6 * max): 7
Non-estimable combinations: 30


## Predicted vs. measured parity

All conditions and outputs (metabolites + fluxes) on one parity plot.  Points
on the 1:1 diagonal are perfectly predicted.  Systematic bias (all points above
or below the diagonal) indicates a structural model deficiency; scattered points
indicate parameter uncertainty or measurement noise.


In [8]:
pred_init = res.get("predictions_init")
pred_fit = res.get("predictions_fitted")
real_df = res.get("real")

if pred_init is not None and real_df is not None:
    fig, ax = gp.plot_pred_vs_meas(
        pred_init, real_df,
        title="Predicted vs. measured (initial params)",
        savepath=os.path.join(FIGURES_DIR, "parity_init.png")
    )
    print("Figure saved: parity_init.png")

if pred_fit is not None and real_df is not None:
    fig, ax = gp.plot_pred_vs_meas(
        pred_fit, real_df,
        title="Predicted vs. measured (fitted params)",
        savepath=os.path.join(FIGURES_DIR, "parity_fitted.png")
    )
    print("Figure saved: parity_fitted.png")

Figure saved: parity_init.png


## Residuals per output

Shows which individual metabolites and fluxes are fitted well vs. poorly.
Large mean residuals after fitting point to structural model limitations
(e.g. missing regulatory interactions or wrong stoichiometry).


In [9]:
if pred_init is not None and real_df is not None:
    fig, ax = gp.plot_residuals(
        pred_init, real_df,
        title="Residuals at initial parameters (pred - meas)",
        savepath=os.path.join(FIGURES_DIR, "residuals_init.png")
    )
    print("Figure saved: residuals_init.png")

if pred_fit is not None and real_df is not None:
    fig, ax = gp.plot_residuals(
        pred_fit, real_df,
        title="Residuals at fitted parameters (pred - meas)",
        savepath=os.path.join(FIGURES_DIR, "residuals_fitted.png")
    )
    print("Figure saved: residuals_fitted.png")

Figure saved: residuals_init.png


## Sensitivity heatmap

The sensitivity matrix G = d(outputs)/d(params) shows which parameters influence
which outputs.  Columns with all near-zero sensitivity are non-influential and
are candidates to fix (at their literature value or the estimated value).

Combined with the FIM (which aggregates G^T Q^-1 G over conditions and weights by
measurement noise), the sensitivity heatmap gives the per-condition view.


In [10]:
sens_dict = res.get("sensitivity")

if sens_dict is not None:
    # Plot the first condition by default.
    first_cond = next(iter(sens_dict))
    fig, ax = gp.plot_sensitivity_heatmap(
        sens_dict,
        condition=first_cond,
        log_scale=True,
        savepath=os.path.join(FIGURES_DIR, "sensitivity_%s.png" % first_cond)
    )
    print("Figure saved: sensitivity_%s.png" % first_cond)

    # Max sensitivity per parameter (across all outputs and conditions).
    max_sens = {}
    for cond, G_df in sens_dict.items():
        G = G_df.to_numpy(dtype=float)
        for j, col in enumerate(G_df.columns):
            max_sens[col] = max(max_sens.get(col, 0.0), float(np.nanmax(np.abs(G[:, j]))))

    max_sens_s = pd.Series(max_sens).sort_values(ascending=False)
    print("\nTop-10 parameters by max |sensitivity| across conditions:")
    print(max_sens_s.head(10).round(4).to_string())
    print("\nBottom-10 (least influential):")
    print(max_sens_s.tail(10).round(6).to_string())
else:
    print("Sensitivity dict not found.")

Figure saved: sensitivity_KO02.png

Top-10 parameters by max |sensitivity| across conditions:
Ks_adp_7    2.609214e+06
Ks_atp_7    6.250368e+05
Ks_f6p_3    5.325282e+05
Ks_atp_3    2.662641e+05
Ks_3pg_7    1.915781e+05
Ks_2pg_9    1.758377e+05
Ks_pgp_7    1.711320e+05
Kp_fbp_3    9.111625e+04
Ks_pep_9    5.087103e+04
Kp_g3p_4    4.207548e+04

Bottom-10 (least influential):
Ks_nad_6     82.971932
kcat_f_8     74.138585
Ks_dhap_5    17.087526
kcat_f_6     12.499015
Kp_nadh_6     9.200144
Ks_pi_6       7.044787
Kp_pgp_6      4.818940
Ks_g3p_6      4.195210
Kp_g3p_5      1.953427
kcat_f_5      0.026040


## Structural identifiability report

The structural report provides three tables:
- `per_condition`: operating-point conditioning (rank, condition number of the
  balanced Jacobian and G matrix) per condition. A rank < 9 or very high condition
  number signals numerical difficulties with the steady-state solve at this operating
  point.
- `identifiability`: aggregate FIM-level summary (FIM rank, number of identifiable
  and non-estimable parameters).
- `per_parameter`: Cramer-Rao standard deviation, coefficient of variation, and
  estimability flag for each free parameter.

**What to look for:** parameters with `identifiable=False` and low `subspace_proj`
are the best candidates to fix in the reduced model.  Parameters with
high `cv_percent` are uncertain even if technically identifiable.


In [11]:
import os

struct_per_cond = None
struct_ident = None
struct_per_param = None

# Load structural report from saved CSVs.
for suffix, attr in [
    ("per_condition", "struct_per_cond"),
    ("identifiability", "struct_ident"),
    ("per_parameter", "struct_per_param"),
]:
    fpath = os.path.join(RESULTS_DIR, "structural_report_%s.csv" % suffix)
    if os.path.exists(fpath):
        exec("%s = pd.read_csv(fpath, index_col=0)" % attr)
        print("Loaded structural_report_%s.csv" % suffix)

if struct_per_cond is not None:
    print("\nPer-condition:")
    print(struct_per_cond.round(2).to_string())

if struct_ident is not None:
    print("\nIdentifiability summary:")
    print(struct_ident.to_string())

if struct_per_param is not None:
    print("\nPer-parameter (sorted by subspace_proj desc, head 15):")
    _cols = [c for c in ["std", "cv_percent", "identifiable",
                          "subspace_proj", "max_abs_sens"]
             if c in struct_per_param.columns]
    _sort_by = "subspace_proj" if "subspace_proj" in struct_per_param.columns else struct_per_param.columns[0]
    print(struct_per_param[_cols].sort_values(_sort_by, ascending=False).head(15).round(3).to_string())

Loaded structural_report_per_condition.csv
Loaded structural_report_identifiability.csv
Loaded structural_report_per_parameter.csv

Per-condition:
           ss_residual  rank_A        cond_A  n_bound_active  rank_G
condition                                                           
KO02              2.50       9  9.685598e+09               0      10
KO03              1.95       9  5.595393e+09               0      11
KO04              0.47       9  5.084638e+06               0       9
KO05              1.68       9  6.732836e+09               0      11
KO07              2.00       9  2.493403e+09               0      10
KO08              1.67       9  1.529354e+09               0      10
KO10            110.96       9  5.156723e+06               0       9
KO11              2.98       9  2.811204e+09               0      11
KO12              2.16       9  3.104819e+09               0      11
KO13              1.67       9  3.085726e+09               0      10
KO14              2.17   

## Interpretation

### Parameter identifiability

The FIM eigenvalue spectrum (see `fim_spectrum.png`) reveals the effective rank
of the parameter estimation problem.  If many eigenvalues are near zero, the 37
free-parameter problem is ill-conditioned: the data constrain only a subspace of
parameter combinations.

From the structural report's `identifiability` table, parameters flagged
`identifiable=False` cannot be reliably estimated from the current Ishii dataset
and should be fixed in the next reduced model.

### Correlation structure

High |r_ij| (> 0.9) in the correlation matrix (see `correlation_heatmap.png`)
identifies collinear parameter pairs.  From each such pair, one parameter should
be fixed (typically the one with lower sensitivity or higher CV%).  Common
collinear patterns in glycolytic models:
- kcat and Km of the same enzyme are anti-correlated (product stays constant at
  the same flux level).
- Product inhibition constants (Kp) are often collinear with the forward Km
  because the thermodynamic constraint links them.

### RMSE: init vs. fitted

A substantial drop in `data_norm` RMSE after fitting (see `rmse_summary.png`)
confirms that the optimizer found a meaningfully better parameter set.  A small
drop indicates either a good initial guess or a very flat objective landscape
(consistent with many near-zero FIM eigenvalues).

### Next steps: fix-and-reduce

This analysis feeds directly into the next step of the estimation workflow:

1. Fix the highly-correlated and low-sensitivity parameters identified above
   (e.g. using `fixed_params=REPORT_FIXED_PARAMS` in the estimator, or a custom
   list from the structural report).
2. Re-run the estimator on the reduced (identifiable) parameter set.
3. Check that RMSE does not increase significantly (the fixed parameters should
   have negligible effect on the fit quality).
4. Optionally, run multistart on the reduced model to verify global optimality.
